# 特征工程实战 - 综合案例

本笔记本演示如何在实际案例中综合应用各种特征工程技术。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("特征工程实战库已导入")

## 1. 数据准备

In [ ]:
# 创建真实的客户数据集
np.random.seed(42)
n_samples = 200

data = pd.DataFrame({
    '年龄': np.random.randint(18, 80, n_samples),
    '年收入': np.random.randint(20000, 150000, n_samples),
    '信用评分': np.random.randint(300, 850, n_samples),
    '既有贷款数': np.random.randint(0, 5, n_samples),
    '过期次数': np.random.randint(0, 10, n_samples),
    '工作年限': np.random.randint(0, 50, n_samples),
    '家庭成员数': np.random.randint(1, 7, n_samples)
})

# 创建目标变量（是否批准贷款）
y = ((data['信用评分'] > 650) & (data['年收入'] > 50000) & (data['过期次数'] < 3)).astype(int)

print("原始数据:")
print(data.head(10))
print()
print(f"数据形状: {data.shape}")
print(f"目标变量分布:\n{y.value_counts()}")

## 2. 特征工程处理

In [ ]:
# 创建特征工程数据框
df_engineered = data.copy()

print("应用特征工程...\n")

# 1. 处理缺失值（这里没有，但展示处理方法）
# df_engineered = df_engineered.fillna(df_engineered.mean())

# 2. 创建交互特征
df_engineered['收入_年龄_交互'] = df_engineered['年收入'] * df_engineered['年龄']

# 3. 创建比率特征
df_engineered['负债比'] = df_engineered['既有贷款数'] / (df_engineered['年收入'] / 10000 + 1)
df_engineered['过期率'] = df_engineered['过期次数'] / (df_engineered['既有贷款数'] + 1)

# 4. 创建分箱特征
df_engineered['年龄段'] = pd.cut(df_engineered['年龄'], 
                                bins=[0, 30, 45, 60, 100],
                                labels=[0, 1, 2, 3])

df_engineered['收入等级'] = pd.cut(df_engineered['年收入'],
                                  bins=[0, 50000, 80000, 120000, 200000],
                                  labels=[0, 1, 2, 3])

# 5. 创建数学变换特征
df_engineered['信用评分_正规化'] = (df_engineered['信用评分'] - 300) / (850 - 300)
df_engineered['年收入_log'] = np.log1p(df_engineered['年收入'])

# 6. 创建布尔特征
df_engineered['是否有过期'] = (df_engineered['过期次数'] > 0).astype(int)
df_engineered['是否多重贷款'] = (df_engineered['既有贷款数'] > 2).astype(int)

print("工程特征:")
print(df_engineered.head(10))
print()
print(f"原始特征数: {data.shape[1]}")
print(f"工程特征数: {df_engineered.shape[1]}")

## 3. 特征标准化

In [ ]:
# 选择数值特征
numeric_features = df_engineered.select_dtypes(include=['int64', 'float64']).columns

# 标准化
scaler = StandardScaler()
df_scaled = df_engineered.copy()
df_scaled[numeric_features] = scaler.fit_transform(df_engineered[numeric_features])

print("标准化后的特征统计:")
print(df_scaled[numeric_features].describe())

## 4. 特征选择

In [ ]:
# 使用 SelectKBest 选择前 8 个特征
X = df_scaled[numeric_features]

selector = SelectKBest(score_func=f_classif, k=8)
X_selected = selector.fit_transform(X, y)

# 获取特征重要性
feature_scores = pd.DataFrame({
    '特征': numeric_features,
    'F值': selector.scores_
}).sort_values('F值', ascending=False)

print("特征重要性排序:")
print(feature_scores)
print()

selected_features = numeric_features[selector.get_support()].tolist()
print(f"选中的特征: {selected_features}")

## 5. 模型对比

In [ ]:
# 准备数据
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, test_size=0.2, random_state=42
)

print(f"训练集大小: {X_train.shape[0]}, 测试集大小: {X_test.shape[0]}")
print()

In [ ]:
# 模型1: 逻辑回归
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

# 模型2: 随机森林
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

# 性能评估
metrics = pd.DataFrame({
    '逻辑回归': [
        accuracy_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_lr)
    ],
    '随机森林': [
        accuracy_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_rf)
    ]
}, index=['准确率', '精准率', '召回率', 'F1分数'])

print("模型性能对比:")
print(metrics)
print()
print(f"测试集准确率 (逻辑回归): {metrics.loc['准确率', '逻辑回归']:.4f}")
print(f"测试集准确率 (随机森林): {metrics.loc['准确率', '随机森林']:.4f}")

## 6. 特征重要性分析

In [ ]:
# 随机森林特征重要性
rf_importance = pd.DataFrame({
    '特征': selected_features,
    '重要性': rf.feature_importances_
}).sort_values('重要性', ascending=False)

print("随机森林特征重要性:")
print(rf_importance)

## 7. 可视化分析

In [ ]:
# 绘制分析结果
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 特征重要性
axes[0, 0].barh(feature_scores['特征'][:10], feature_scores['F值'][:10],
                 color='skyblue', edgecolor='navy')
axes[0, 0].set_xlabel('F值分数')
axes[0, 0].set_title('SelectKBest 特征重要性 (Top 10)')
axes[0, 0].grid(True, alpha=0.3, axis='x')

# 随机森林特征重要性
axes[0, 1].barh(rf_importance['特征'], rf_importance['重要性'],
                 color='lightgreen', edgecolor='darkgreen')
axes[0, 1].set_xlabel('重要性')
axes[0, 1].set_title('随机森林特征重要性')
axes[0, 1].grid(True, alpha=0.3, axis='x')

# 模型性能对比
metrics.T.plot(kind='bar', ax=axes[1, 0])
axes[1, 0].set_title('模型性能对比')
axes[1, 0].set_ylabel('分数')
axes[1, 0].set_ylim([0, 1])
axes[1, 0].grid(True, alpha=0.3, axis='y')
axes[1, 0].legend(loc='lower right')
axes[1, 0].set_xticklabels(axes[1, 0].get_xticklabels(), rotation=45)

# 目标变量分布
y.value_counts().plot(kind='bar', ax=axes[1, 1], color=['lightcoral', 'lightblue'],
                      edgecolor='black')
axes[1, 1].set_title('目标变量分布')
axes[1, 1].set_ylabel('数量')
axes[1, 1].set_xticklabels(['拒绝(0)', '批准(1)'], rotation=0)
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 8. 特征工程影响分析

In [ ]:
# 对比原始特征 vs 工程特征的模型性能
print("特征工程的影响分析:\n")

# 使用原始特征
X_raw = data.copy()
X_raw_train, X_raw_test, y_raw_train, y_raw_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42
)

rf_raw = RandomForestClassifier(n_estimators=100, random_state=42)
rf_raw.fit(X_raw_train, y_raw_train)
y_pred_raw = rf_raw.predict(X_raw_test)

# 对比
print(f"原始特征准确率: {accuracy_score(y_raw_test, y_pred_raw):.4f}")
print(f"工程特征准确率: {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"\n性能提升: {(accuracy_score(y_test, y_pred_rf) - accuracy_score(y_raw_test, y_pred_raw)):.4f}")